# JaxMARL-BC quickstart

Load a config, train a short experiment, inspect the economic diagnostics, and run a mini scaling sweep — all from the standard package API.

On Colab T4 set `run.device=auto` (the default). Here we use `cpu` with tiny budgets so the notebook runs anywhere.

In [ ]:
from jmbc.config import load_config, setup_device
setup_device('auto')  # call before importing jax-heavy modules; picks the GPU on Colab

cfg = load_config('rbc', [
    'env.delta=1.0',            # textbook RBC has a closed-form solution
    'train.total_timesteps=12500',   # sequential steps, independent of num_envs
    'train.rollout_len=100', 'train.num_envs=16', 'train.num_minibatches=4',
    'env.max_steps=200', 'diag.sim_steps=1500', 'diag.n_snapshots=4',
])
print(cfg.exp, '| n_agents', cfg.env.n_agents, '| delta', cfg.env.delta)

In [ ]:
from jmbc.config import load_config, setup_device
setup_device('cpu')  # call before importing jax-heavy modules

cfg = load_config('rbc', [
    'env.delta=1.0',            # textbook RBC has a closed-form solution
    'train.total_timesteps=200000',
    'train.rollout_len=100', 'train.num_envs=16', 'train.num_minibatches=4',
    'env.max_steps=200', 'diag.sim_steps=1500', 'diag.n_snapshots=4',
])
print(cfg.exp, '| n_agents', cfg.env.n_agents, '| delta', cfg.env.delta)

In [ ]:
from jmbc.experiments.common import run_single
res = run_single(cfg, do_figures=False)
t = res['timing']
print(f"trained in {t['wall_time_s']:.1f}s  |  {t['throughput_steps_per_s']:.0f} env-steps/s")

## Economic diagnostics

Does the trained policy behave like a correct economy? For textbook RBC we can compare to the analytical solution and check the Euler equation and the resource-constraint identity.

In [ ]:
import json
econ = res['summary']['final']['economic']
print('Euler error (mean |residual|):', round(econ['euler']['euler_mean_abs'], 4))
print('Resource residual (relative) :', round(econ['resource']['resource_mean_rel'], 5))
print('Analytical comparison:')
print(json.dumps({k: round(v, 4) for k, v in econ['analytical_rbc'].items()}, indent=2))

In [ ]:
# Economic accuracy improving over training (Euler error vs steps).
from jmbc.plots import plot_economic_snapshots
plot_economic_snapshots(res['summary'], 'rbc', res['steps_per_update'], '_econ.png')
from IPython.display import Image
Image('_econ.png')

import pandas as pd
from jmbc.recorder import benchmark_time
from jmbc.algos import make_train
from jmbc.envs import build_env
from jmbc.config import to_train_dict
import jax

rows = []
for n_agents in [1, 10, 100]:
    c = load_config('rbc', [f'env.n_agents={n_agents}', 'train.total_timesteps=2500',
                            'train.rollout_len=50', 'train.num_envs=16', 'env.max_steps=100'])
    train_fn = make_train(build_env(c.env), to_train_dict(c))
    _, timing = benchmark_time(train_fn, jax.random.PRNGKey(0))
    rows.append({'n_agents': n_agents, **timing})
df = pd.DataFrame(rows)
df[['n_agents', 'run_only_s', 'throughput_steps_per_s']]

In [ ]:
import pandas as pd
from jmbc.recorder import benchmark_time
from jmbc.algos import make_train
from jmbc.envs import build_env
from jmbc.config import to_train_dict
import jax

rows = []
for n_agents in [1, 10, 100]:
    c = load_config('rbc', [f'env.n_agents={n_agents}', 'train.total_timesteps=40000',
                            'train.rollout_len=50', 'train.num_envs=16', 'env.max_steps=100'])
    train_fn = make_train(build_env(c.env), to_train_dict(c))
    _, timing = benchmark_time(train_fn, jax.random.PRNGKey(0))
    rows.append({'n_agents': n_agents, **timing})
df = pd.DataFrame(rows)
df[['n_agents', 'run_only_s', 'throughput_steps_per_s']]

In [ ]:
from jmbc.plots import plot_metric_vs
df['method'] = 'jaxmarl-bc'
plot_metric_vs(df, 'n_agents', 'throughput_steps_per_s', '_scaling.png',
               ylabel='env steps / s', title='Throughput vs agents')
Image('_scaling.png')